# Pyramid construction scaling

Time `coarsen_level` for **point clouds** (vertex-only) and **graphs**
(N nodes + edges) at increasing vertex count `N`, comparing three
uniform coarsening ratios `R ∈ {2, 4, 8}` over 3 levels (resulting
pyramids span level 0 / 1 / 2 / 3).

Each `(N, R, geometry)` cell is averaged over **`N_RUNS = 10` runs**;
the plot shows the mean with a shaded **95% confidence interval**
(Student's t, df=9).

Per-level timings come from `coarsen_level(source, target,
coarsen_factor=R, sparsity_factor=1.0)` called in a loop — same
work as `build_pyramid(factors=[(R, 1)]*3)` but with one timing per
hop. `coarsen_level`'s default `cross_level_storage="none"` isolates
raw coarsening cost; full `build_pyramid` runs with
`cross_level_storage="explicit"` and is moderately slower.

Runtime: ~15–30 minutes on a laptop (the 1 M tier dominates).

In [ ]:
import os, time, tempfile, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def _time(fn, *args, **kwargs):
    """Call fn(*args, **kwargs); return (elapsed_seconds, result)."""
    t0 = time.perf_counter()
    out = fn(*args, **kwargs)
    return time.perf_counter() - t0, out


def _store_bytes(path):
    """Total on-disk size of a store directory, in bytes."""
    p = Path(path)
    return sum(f.stat().st_size for f in p.rglob('*') if f.is_file())


def _new_store(prefix):
    """Fresh tempdir + zarrvectors path."""
    return Path(tempfile.mkdtemp(prefix=f'zvbench_{prefix}_')) / 'store.zarrvectors'


N_RUNS = 10
T95_DF9 = 2.262  # scipy.stats.t.ppf(0.975, df=9) — hard-coded to avoid scipy dep


def _mean_ci95(samples):
    """(mean, half-width) for a 1-D sample using Student's t, df=n-1."""
    arr = np.asarray(samples, dtype=float)
    if arr.size < 2:
        return float(arr.mean()) if arr.size else 0.0, 0.0
    m = arr.mean()
    s = arr.std(ddof=1)
    hw = T95_DF9 * s / np.sqrt(arr.size)
    return float(m), float(hw)

## 1 · Setup

In [ ]:
from zarr_vectors.types.points import write_points
from zarr_vectors.types.graphs import write_graph
from zarr_vectors.multiresolution.coarsen import coarsen_level

SIZES  = [10_000, 100_000, 1_000_000]
RATIOS = [2, 4, 8]
N_LVLS = 3                    # builds levels 1, 2, 3 from level 0
CHUNK  = (200.0, 200.0, 200.0)
BIN    = (50.0,  50.0,  50.0)
SEED   = 0

## 2 · Run the sweep

In [ ]:
rng = np.random.default_rng(SEED)

metrics = ['lvl1_s', 'lvl2_s', 'lvl3_s', 'total_s']
# raw[metric][(geom, ratio)] is shape (len(SIZES), N_RUNS)
raw = {
    m: {
        (g, r): np.zeros((len(SIZES), N_RUNS))
        for g in ('points', 'graph')
        for r in RATIOS
    }
    for m in metrics
}

for i, n in enumerate(SIZES):
    positions = rng.uniform(0, 1000, (n, 3)).astype(np.float32)
    # ~3N/2 random undirected edges, dedup'd against self-loops.
    src = rng.integers(0, n, size=3 * n // 2)
    dst = rng.integers(0, n, size=3 * n // 2)
    mask = src != dst
    edges = np.stack([src[mask], dst[mask]], axis=1).astype(np.int64)

    geometries = [
        ('points', write_points, (positions,)),
        ('graph',  write_graph,  (positions, edges)),
    ]

    for ratio in RATIOS:
        for run in range(N_RUNS):
            for geom, write_fn, args in geometries:
                store = _new_store(f'pyr_{geom}_n{n}_r{ratio}_run{run}')
                write_fn(store, *args, chunk_shape=CHUNK, bin_shape=BIN)
                per_lvl = []
                for lvl in range(N_LVLS):
                    t, _ = _time(
                        coarsen_level, store,
                        source_level=lvl, target_level=lvl + 1,
                        coarsen_factor=float(ratio),
                        sparsity_factor=1.0,
                    )
                    per_lvl.append(t)
                raw['lvl1_s'][(geom, ratio)][i, run]  = per_lvl[0]
                raw['lvl2_s'][(geom, ratio)][i, run]  = per_lvl[1]
                raw['lvl3_s'][(geom, ratio)][i, run]  = per_lvl[2]
                raw['total_s'][(geom, ratio)][i, run] = sum(per_lvl)
                shutil.rmtree(Path(store).parent, ignore_errors=True)

# Summarise into a tidy dataframe: one row per (N, geom, ratio).
rows = []
for i, n in enumerate(SIZES):
    for geom in ('points', 'graph'):
        for ratio in RATIOS:
            row = {'N': n, 'geom': geom, 'ratio': ratio}
            for m in metrics:
                mean, hw = _mean_ci95(raw[m][(geom, ratio)][i])
                row[f'{m}_mean'] = round(mean, 4)
                row[f'{m}_hw']   = round(hw,   4)
            rows.append(row)

df = pd.DataFrame(rows)

## 3 · Results

In [ ]:
df

## 4 · Plot

In [ ]:
COLORS = {2: 'C0', 4: 'C1', 8: 'C2'}
GEOMS  = ['points', 'graph']

fig, axes = plt.subplots(2, 2, figsize=(12, 7))

for row, geom in enumerate(GEOMS):
    # Left column: total build time vs N (one line per ratio, log-log).
    ax = axes[row, 0]
    for ratio in RATIOS:
        sub  = df[(df.geom == geom) & (df.ratio == ratio)].sort_values('N')
        mean = sub['total_s_mean'].values
        hw   = sub['total_s_hw'].values
        ax.fill_between(
            sub['N'], mean - hw, mean + hw, color=COLORS[ratio], alpha=0.2,
        )
        ax.loglog(
            sub['N'], mean, 'o-', color=COLORS[ratio], label=f'R={ratio}',
        )
    ax.set_xlabel('N (vertices)')
    ax.set_ylabel('s')
    ax.set_title(f'{geom}: total build time vs N')
    ax.grid(True, which='both', alpha=0.3)

    # Right column: per-level breakdown at the largest N (grouped bar).
    ax = axes[row, 1]
    x_pos = np.arange(N_LVLS)
    for j, ratio in enumerate(RATIOS):
        sub = df[(df.geom == geom)
                 & (df.ratio == ratio)
                 & (df.N == max(SIZES))].iloc[0]
        means = [sub[f'lvl{k + 1}_s_mean'] for k in range(N_LVLS)]
        hws   = [sub[f'lvl{k + 1}_s_hw']   for k in range(N_LVLS)]
        ax.bar(
            x_pos + (j - 1) * 0.27, means, width=0.27,
            yerr=hws, capsize=3, color=COLORS[ratio], label=f'R={ratio}',
        )
    ax.set_xticks(x_pos)
    ax.set_xticklabels(['L0→L1', 'L1→L2', 'L2→L3'])
    ax.set_ylabel('s')
    ax.set_title(f'{geom}: per-level breakdown @ N={max(SIZES):,}')
    ax.grid(True, axis='y', alpha=0.3)

axes[0, 0].legend(loc='best')
fig.suptitle(
    f'Pyramid construction scaling ({N_RUNS} runs, 95% CI)',
)
plt.tight_layout()